In [28]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder
from sklearn.preprocessing import OneHotEncoder
from mlxtend.frequent_patterns import association_rules
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display

In [29]:
df = pd.read_csv("../Datasets/Housing.csv")
df.head()

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no


In [30]:
intervals = [(0,50000),(50000,100000),(100000,150000),(150000,200000)]
bins = pd.IntervalIndex.from_tuples(intervals)
df['price_slab'] = pd.cut(df['price'], bins)

In [31]:
intervals = [(0,5000),(5000,10000),(10000,15000),(15000,20000)]
bins = pd.IntervalIndex.from_tuples(intervals)
df['area_slab'] = pd.cut(df['lotsize'], bins)

In [32]:
df.drop(['price','lotsize'],axis=1,inplace=True)

In [33]:
ohc = OneHotEncoder(sparse_output=False, dtype=np.bool_).set_output(transform='pandas')
df = ohc.fit_transform(df)

In [34]:
def gen_rules (min_support, min_confidence):
    itemsets = apriori(df, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)
    rules = rules[['antecedents','consequents','support','confidence','lift']]
    rel_rules = rules[rules["lift"]>1]
    return rel_rules.sort_values(by=['confidence','lift','support'],ascending=False)

In [35]:
interact(
    gen_rules, 
    min_support = widgets.FloatSlider(value=0.01, min=0, max=1, step=0.01, description='Min Support:'), 
    min_confidence = widgets.FloatSlider(value=0.01, min=0, max=1, step=0.01, description='Min Confidence:')
)

interactive(children=(FloatSlider(value=0.01, description='Min Support:', max=1.0, step=0.01), FloatSlider(val…

<function __main__.gen_rules(min_support, min_confidence)>

In [36]:
gen_rules (0.6,0.7)

,antecedents,consequents,support,confidence,lift
23,"(bathrms_1, driveway_yes)",(gashw_no),0.602564,0.964809,1.011105
4,(bathrms_1),(gashw_no),0.708791,0.962687,1.008881
28,"(bathrms_1, recroom_no)",(gashw_no),0.602564,0.959184,1.005210
9,(driveway_yes),(gashw_no),0.820513,0.955224,1.001060
34,"(recroom_no, driveway_yes)",(gashw_no),0.663004,0.955145,1.000977
18,(fullbase_no),(gashw_no),0.620879,0.954930,1.000752
12,(fullbase_no),(recroom_no),0.602564,0.926761,1.126974
8,(gashw_no),(driveway_yes),0.820513,0.859885,1.001060
38,"(gashw_no, prefarea_no)",(recroom_no),0.622711,0.858586,1.044071
17,(prefarea_no),(recroom_no),0.655678,0.856459,1.041485
